# Chapter 4 (part 2) — Factor (Categorical) Variables in Regression

Covers: Dummy Variable Representation → Factor Variables with Many Levels → Ordered Factor Variables, on Ames Housing.

The thread: **a regression only understands numbers, so every category gets turned into numbers by *some* coding, and the coding decides what the coefficients mean.** Dummies turn "which group" into "how far from a baseline group". Many-level factors leave rare levels with almost no data behind their coefficients. Ordered factors tempt you into integers, which quietly assume equal gaps between levels.

Data note: quality ratings here are *words* (`Overall_Qual` is `Very_Poor` … `Very_Excellent`, not 1-10), so you'll write the integer mapping yourself.

**How to work each section** (~10 min each; ~1 hour total with the exam): write your one-line prediction in the *Prediction* cell **before** running anything, do the code, then write 2-3 bullets on where the result matched or broke your prediction and why. Ask for a hint if you get stuck.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

ames = pd.read_csv('../data/ames_housing.csv')
ames.shape

## 1. Dummy Variables and the Reference Level

To feed a category into a regression you make one 0/1 column per level and leave one level out as the baseline. Numeric picture: average prices are $200k for single-family, $150k for duplexes, $140k for townhouses. With single-family as the baseline, the intercept is $200k, the duplex coefficient is -$50k, the townhouse coefficient is -$60k, and a duplex's prediction is 200 + (-50) = $150k, exactly its group mean. Switch the baseline to townhouse: the intercept becomes $140k, single-family +$60k, duplex +$10k, and the three predictions are still 200, 150, 140. The baseline is a choice about which comparison is easiest to read, not a modeling decision.

**Predict first:** with only `Bldg_Type` in the model, is R² closer to 0.03, 0.15, or 0.40? When you change the baseline, which of these change: the coefficients, R², the fitted values?

**Question 1.** (a) Fit `Sale_Price ~ C(Bldg_Type)`. Name the baseline level and check against the group means that intercept = baseline mean and each coefficient = group mean − baseline mean. (b) Some levels have much bigger standard errors than others. Why? (c) Re-fit with `TwnhsE` as the baseline and confirm which numbers changed and which stayed identical.

*Tools for this question:*
- `C(col)` inside a formula treats `col` as categorical (one dummy per level except the first alphabetically). Coefficient names look like `C(Bldg_Type)[T.Duplex]`.
- `C(col, Treatment(reference="TwnhsE"))` sets the baseline level.
- `df.groupby("col")["y"].mean()` for group means; `df["col"].value_counts()` for group sizes.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 2. Dummies Alongside a Numeric Predictor

Once a numeric predictor is in the model, a dummy coefficient no longer compares *group averages*; it compares groups **at the same value of the numeric predictor**. Numeric picture: duplexes average $150k at 2,000 sqft; single-family homes average $200k at 1,500 sqft. Raw gap = -$50k. But the duplexes are bigger; if each sqft is worth $100, a 2,000 sqft house "should" be worth $50k more, so *at equal size* the duplex gap is not -$50k but -$100k. Same coefficient label, different question.

**Predict first:** are townhouses (`Twnhs`, `TwnhsE`) smaller or larger than single-family homes on average? Given that, will their coefficients move up or down once `Gr_Liv_Area` is in the model?

**Question 2.** (a) Fit `Sale_Price ~ Gr_Liv_Area + C(Bldg_Type)` and compare each `Bldg_Type` coefficient to Question 1. Which moved most, and in which direction? Explain it with the mean `Gr_Liv_Area` by `Bldg_Type`. (b) In one sentence: what question does the `Duplex` coefficient answer now that it didn't before?

*Tools for this question:*
- `df.groupby("Bldg_Type")["Gr_Liv_Area"].mean()`.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 3. Factor Variables with Many Levels

`Neighborhood` has 28 levels, from North Ames (443 homes) down to Landmark (1 home). A dummy for a one-house neighborhood fits that house *perfectly*: its coefficient is exactly that house's price gap, with nothing to tell signal from noise. The book's remedy is to consolidate levels: fit a model without the factor, look at the residuals grouped by level, and merge levels with similar residuals into a few bins.

Leakage warning: if you compute those bins on all the data and then evaluate on the same data, the grouping has already "seen" the answers. The honest version builds the bins on the *training* split only and applies them to the test split.

**Predict first:** the 28 dummies will beat "no neighborhood" on test RMSE. How much of that gain will the 5 tiers keep: almost none, about half, or nearly all?

**Question 3.** (a) Fit `Sale_Price ~ C(Neighborhood)` and look at the coefficients and standard errors of the five smallest neighborhoods. What do you see? (b) Split train/test 70/30 with a fixed seed. On the **train set only**, fit `Sale_Price ~ Gr_Liv_Area + Year_Built`, take the median residual per neighborhood, and cut the neighborhoods into 5 tiers by those medians. (c) Fit three models on train and compare **test RMSE**: no neighborhood information, 28 neighborhood dummies, the 5-tier factor (all with `Gr_Liv_Area + Year_Built`). Which wins, and why?

*Tools for this question:*
- `df["Neighborhood"].value_counts()`; `df.assign(resid=res.resid).groupby("Neighborhood")["resid"].median()`.
- `pd.qcut(series, q=5, labels=False)` cuts into 5 equal-count bins; `df["col"].map(dict_or_series)` attaches the tiers.
- Gotcha: a neighborhood that's in test but not in train makes `res.predict(test)` raise a confusing `PatsyError`, and maps to `NaN` in the tier column. Drop those rows first: `test[test["Neighborhood"].isin(train["Neighborhood"].unique())]`.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 4. Ordered Factor Variables

`Overall_Qual` runs from `Very_Poor` to `Very_Excellent`, ten ordered words. Two natural encodings: integers 1-10 (one slope: "each step up in quality is worth $X") or dummies (nine separate steps). Numeric picture: if the integer model says each step is worth $30k, it predicts the same $30k gap from Poor to Fair as from Excellent to Very_Excellent. The dummy version would let those gaps be $5k and $100k if that's what the data says, at the cost of eight extra parameters.

**Predict first:** as quality goes up, will the dollar gap from one level to the next grow, shrink, or stay roughly constant?

**Question 4.** (a) Map the ten level names to 1-10 and fit `Sale_Price ~ qual_int` and `Sale_Price ~ C(Overall_Qual)`. Plot the mean price per quality level against the integer model's straight line. Where does the line fail? (b) Compute the 9 step sizes (mean price at level k+1 minus level k). Are they equal? (c) Which model would you use for a one-sentence stakeholder summary, and which for the best predictions?

*Tools for this question:*
- `series.map({"Very_Poor": 1, "Poor": 2, ...})` for the integer coding (check exact spellings with `value_counts()`); Group by your integer column, not the word column: `df.groupby("qual_int")["Sale_Price"].mean().diff()`. Grouping by `Overall_Qual` sorts the words alphabetically (`Above_Average`, `Average`, `Below_Average`, ...), which scrambles both the step sizes and the plot.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

---
# Exam — check your intuition

5 questions, about 3-4 minutes each. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, 2-3 bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Reading dummy coefficients
*Role lean: DS*

Here is `Sale_Price ~ C(Foundation)` on all Ames homes. (`statsmodels` silently picked `BrkTil` as the reference level because it comes first alphabetically.)

```text
                             coef  std err     t   P>|t|
Intercept                128107.0   3847.0  33.0  <0.001
C(Foundation)[T.CBlock]   20177.0   4301.0   5.0  <0.001
C(Foundation)[T.PConc]    98962.0   4279.0  23.0  <0.001
C(Foundation)[T.Slab]    -17650.0  10427.0  -2.0   0.091
C(Foundation)[T.Stone]    21680.0  20812.0   1.0   0.298
C(Foundation)[T.Wood]     52793.0  30581.0   2.0   0.084

R-squared = 0.280
Homes per foundation: PConc 1310, CBlock 1244, BrkTil 311, Slab 49, Stone 11, Wood 5
```

(a) What is the average sale price of **Slab** homes, and how do you know from the table alone? (b) Slab's coefficient is -17,650 and Wood's is +52,793, yet *neither* has a small p-value while CBlock's +20,177 does. How can a difference of similar or larger size (Wood) be "less significant"? (c) Without re-fitting: if we change the reference level to `PConc`, what will the new intercept be, what will the `CBlock` coefficient become, and what will R-squared be? (d) Would changing the reference level change any *prediction* the model makes? Why do people still care which level is the reference?

**Your answer (Q1):**

> _write here_

**Review (Q1):**

> _pending — filled in after you answer_

### Exam Q2 — The dummy-variable trap
*Role lean: DS · ML · interview*

Foundation has 6 levels. (a) Suppose you build 6 dummy columns (one per level) and **also** keep an intercept. Why does ordinary least squares break (or give nonsense) — use a concrete row, e.g. a PConc house, to say what the columns add up to. (b) Suppose instead you keep all 6 dummies and **drop the intercept**. What does each coefficient now equal, and what has to be different about how you read R-squared in that model? (c) Interview version: "Why does one-hot encoding for linear regression drop a column, but a random forest doesn't care?" Answer in two or three sentences.

**Your answer (Q2):**

> _write here_

**Review (Q2):**

> _pending — filled in after you answer_

### Exam Q3 — Factors with many levels
*Role lean: DS · ML*

`Exterior_1st` has 16 levels. This is a slice of `Sale_Price ~ C(Exterior_1st)` showing only the seven rarest levels (reference level is `AsbShng`, 44 homes):

```text
         coef vs AsbShng  std err  homes
ImStucc         160499.0  73799.0    1.0
PreCast         158499.0  73799.0    1.0
CBlock           -8251.0  52761.0    2.0
AsphShn         -10314.0  52761.0    2.0
Stone           156999.0  52761.0    2.0
BrkComm          35999.0  31758.0    6.0
Stucco           48259.0  15648.0   43.0
```

(a) Why are the standard errors so large for the levels with 1-2 homes, and what, concretely, is the model doing to fit a 1-home level perfectly? (b) Name two reasonable ways to handle a factor like this before putting it in a regression (think: what do you do with the rare levels, and what do you do about levels that behave similarly). (c) A colleague proposes replacing the factor with "the mean `Sale_Price` of each exterior type" as one numeric column. Name one big advantage and one serious danger of that approach.

**Your answer (Q3):**

> _write here_

**Review (Q3):**

> _pending — filled in after you answer_

### Exam Q4 — Treating an ordered factor as a number
*Role lean: DS · ML · interview*

`Kitchen_Qual` is ordered (Fair < Typical < Good < Excellent). Someone codes it 1-4 and fits a single slope; you can compare to the model with one dummy per level:

```text
Model: Sale_Price ~ kq   (the single numeric column kq; 1 'Poor' home dropped)
Intercept = -23,456   slope on kq = 81,317   R-squared = 0.454

Kitchen_Qual  coded as  homes  actual mean price  linear-coding fitted
        Fair         1     70             105907                 57861
     Typical         2   1494             139550                139177
        Good         3   1160             210836                220494
   Excellent         4    205             337339                301810

Same data with one dummy per level (C(Kitchen_Qual)): R-squared = 0.482
```

(a) What does the linear coding force to be true about the gaps between adjacent quality levels, and where in the table can you see that it's not true? (b) The dummy model has higher R-squared with 2 extra parameters. Is that an argument for the dummies, against, or neither — and what would you check to decide? (c) Interview version: "For ordinal features, when would you choose integer coding, one-hot, or something else, and does your answer change between linear regression and gradient-boosted trees?" 

**Your answer (Q4):**

> _write here_

**Review (Q4):**

> _pending — filled in after you answer_

### Exam Q5 — Target encoding and leakage (supplementary)
*Role lean: ML · DS · interview*

A high-cardinality factor like `Neighborhood` (28 levels) is converted into one numeric column: each house gets the **mean Sale_Price of its neighborhood**. Two ways to compute those means, each used as the *prediction itself* for 500 held-out homes (averaged over 30 random splits; the 'train-only' means are computed from just 150 homes):

```text
neighborhood means computed from ALL 2,930 homes (including the held-out ones):   held-out R-squared = 0.585
neighborhood means computed only from the 150 training homes:                       held-out R-squared = 0.470
```
(a) Why is the first number higher, and which one tells you how the feature would do on genuinely new data? (b) Why would the gap be even *bigger* if the neighborhoods were smaller (say, 3 homes each)? (c) Describe in two or three sentences how you'd build this feature correctly inside cross-validation, and one extra trick people use to stop small groups from being over-trusted.

**Your answer (Q5):**

> _write here_

**Review (Q5):**

> _pending — filled in after you answer_